In [4]:
# load and check data
import pandas as pd

check_df = pd.read_csv("D:/Projects/e-commerce_funnel_analysis/data/events.csv")
check_df.head()

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2020-09-24 11:57:06 UTC,view,1996170,2144415922528452715,electronics.telephone,NaN,31.90,1515915625519388267,LJuJVLEjPT
1,2020-09-24 11:57:26 UTC,view,139905,2144415926932472027,computers.components.cooler,zalman,17.16,1515915625519380411,tdicluNnRY
2,2020-09-24 11:57:27 UTC,view,215454,2144415927158964449,NaN,NaN,9.81,1515915625513238515,4TMArHtXQy
3,2020-09-24 11:57:33 UTC,view,635807,2144415923107266682,computers.peripherals.printer,pantum,113.81,1515915625519014356,aGFYrNgC08
4,2020-09-24 11:57:36 UTC,view,3658723,2144415921169498184,NaN,cameronsino,15.87,1515915625510743344,aa4mmk0kwQ


In [5]:
# Check Structure
check_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 885129 entries, 0 to 885128
Data columns (total 9 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   event_time     885129 non-null  object 
 1   event_type     885129 non-null  object 
 2   product_id     885129 non-null  int64  
 3   category_id    885129 non-null  int64  
 4   category_code  648910 non-null  object 
 5   brand          672765 non-null  object 
 6   price          885129 non-null  float64
 7   user_id        885129 non-null  int64  
 8   user_session   884964 non-null  object 
dtypes: float64(1), int64(3), object(5)
memory usage: 60.8+ MB


In [6]:
# Check Missing Values
check_df.isnull().sum()

event_time            0
event_type            0
product_id            0
category_id           0
category_code    236219
brand            212364
price                 0
user_id               0
user_session        165
dtype: int64

In [7]:
# Check Duplicates
check_df.duplicated().sum()

np.int64(655)

#### Initial check done.Now Actual data loading from database

In [9]:
import pandas as pd
from db_connection import get_engine
from sqlalchemy import create_engine, text

# Initialize database connection engine
engine = get_engine()

#check connection 
query = "SELECT * FROM events limit 10;"
df = pd.read_sql(query, engine)

# Verify data ingestion
print(f"Rows loaded: {df.shape[0]}")
df.head()

Rows loaded: 10


,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2020-09-24 11:57:06,view,1996170,2144415922528452715,electronics.telephone,None,31.90,1515915625519388267,LJuJVLEjPT
1,2020-09-24 11:57:26,view,139905,2144415926932472027,computers.components.cooler,zalman,17.16,1515915625519380411,tdicluNnRY
2,2020-09-24 11:57:27,view,215454,2144415927158964449,None,None,9.81,1515915625513238515,4TMArHtXQy
3,2020-09-24 11:57:33,view,635807,2144415923107266682,computers.peripherals.printer,pantum,113.81,1515915625519014356,aGFYrNgC08
4,2020-09-24 11:57:36,view,3658723,2144415921169498184,None,cameronsino,15.87,1515915625510743344,aa4mmk0kwQ


In [10]:
# 1. Load full data
df = pd.read_sql("SELECT * FROM events", engine)
print(f"Rows before cleaning: {len(df)}")

Rows before cleaning: 885129


In [11]:
# 2. Remove duplicates
df = df.drop_duplicates(
    subset=['event_time', 'event_type', 'product_id', 'user_id', 'user_session']
)
print(f"Rows after dedup: {len(df)}")

Rows after dedup: 884474


In [12]:
# 3. Parse category_code into 3 levels
df['category_l1'] = df['category_code'].str.split('.').str[0]
df['category_l2'] = df['category_code'].str.split('.').str[1]
df['category_l3'] = df['category_code'].str.split('.').str[2]

In [13]:
# 4. Fix event_time dtype 
df['event_time'] = pd.to_datetime(df['event_time'])

In [14]:
# 5. Standardise text columns
df['brand'] = df['brand'].str.lower().str.strip()
df['event_type'] = df['event_type'].str.lower().str.strip()
df['category_l1'] = df['category_l1'].str.lower().str.strip()
df['category_l2'] = df['category_l2'].str.lower().str.strip()


In [15]:
# 6. Drop rows with null user_session (only 165)
df = df.dropna(subset=['user_session'])
print(f"Rows after dropping null sessions: {len(df)}")

Rows after dropping null sessions: 884312


In [16]:
# 7. Summary check 
print("\n── Null counts after cleaning ──")
print(df.isnull().sum())

print("\n── Event type distribution ──")
print(df['event_type'].value_counts())

print("\n── Category L1 top 10 ──")
print(df['category_l1'].value_counts().head(10))



── Null counts after cleaning ──
event_time            0
event_type            0
product_id            0
category_id           0
category_code    236001
brand            212194
price                 0
user_id               0
user_session          0
category_l1      236001
category_l2      236001
category_l3      435727
dtype: int64

── Event type distribution ──
event_type
view        792943
cart         54026
purchase     37343
Name: count, dtype: int64

── Category L1 top 10 ──
category_l1
computers       316660
electronics     170949
stationery       42948
appliances       41047
auto             35408
construction     31035
furniture         3364
country_yard      3137
accessories       2075
medicine           712
Name: count, dtype: int64


In [17]:
print(f"Total rows: {len(df)}")
print(f"\n── Event type distribution ──")
print(df['event_type'].value_counts())
print(f"\n── Date range ──")
print(f"From: {df['event_time'].min()}")
print(f"To:   {df['event_time'].max()}")
print(f"\n── Unique users ──")
print(f"Users:    {df['user_id'].nunique():,}")
print(f"Products: {df['product_id'].nunique():,}")
print(f"Brands:   {df['brand'].nunique():,}")
print(f"\n── Category L1 top 10 ──")
print(df['category_l1'].value_counts().head(10))

Total rows: 884312

── Event type distribution ──
event_type
view        792943
cart         54026
purchase     37343
Name: count, dtype: int64

── Date range ──
From: 2020-09-24 11:57:06
To:   2021-02-28 23:59:09

── Unique users ──
Users:    407,237
Products: 53,452
Brands:   999

── Category L1 top 10 ──
category_l1
computers       316660
electronics     170949
stationery       42948
appliances       41047
auto             35408
construction     31035
furniture         3364
country_yard      3137
accessories       2075
medicine           712
Name: count, dtype: int64


In [19]:
# Write cleaned data back to PostgreSQL
print("Writing cleaned data to PostgreSQL...")

'''df.to_sql(
    'events_cleaned',
    engine,
    if_exists='replace',
    index=False,
    method='multi',
    chunksize=10000
)

print(f"Done. {len(df):,} rows written to events_clean table.")'''

Writing cleaned data to PostgreSQL...


'df.to_sql(\n    \'events_cleaned\',\n    engine,\n    if_exists=\'replace\',\n    index=False,\n    method=\'multi\',\n    chunksize=10000\n)\n\nprint(f"Done. {len(df):,} rows written to events_clean table.")'

In [20]:
# Write cleaned data back to PostgreSQL
# Step 1 — save as CSV
df.to_csv(
    'D:/Projects/e-commerce_funnel_analysis/data/events_clean.csv', 
    index=False
)
print("CSV saved.")

CSV saved.


In [21]:
# Step 2 — load into PostgreSQL via COPY
from sqlalchemy import text

with engine.connect() as conn:
    conn.execute(text("DROP TABLE IF EXISTS events_clean;"))
    conn.execute(text("""
        CREATE TABLE events_clean (
            event_time      TIMESTAMP,
            event_type      VARCHAR(20),
            product_id      BIGINT,
            category_id     BIGINT,
            category_code   VARCHAR(255),
            brand           VARCHAR(100),
            price           NUMERIC(10,2),
            user_id         BIGINT,
            user_session    TEXT,
            category_l1     VARCHAR(100),
            category_l2     VARCHAR(100),
            category_l3     VARCHAR(100)
        );
    """))
    conn.execute(text("""
        COPY events_clean 
        FROM 'D:/Projects/e-commerce_funnel_analysis/data/events_clean.csv'
        DELIMITER ',' CSV HEADER;
    """))
    conn.commit()

print("Done!")

Done!
